In [1]:
import numpy as np
import os
import itertools
import logging
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
import torch.nn.functional as F
from argparse import ArgumentParser
from torch.distributions import MultivariateNormal
import sys
sys.path.append("../")
from config import get_cfg_defaults
from nf.flows import *
from nf.models import NormalizingFlowModel
from nf.base import EinsteinCrystal
import nf.utils as util
import random
import systems
import pymbar
from bar import BAR
sys.path.append("/home/sherryli/xsli/MBAR")
from mbar.solve import solver
%load_ext autoreload
%autoreload 2

/home/sherryli/xsli/softwares/anaconda3/envs/sherry/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def setup_model(cfg):
    if cfg.dataset.rho is not None:
        B=(cfg.dataset.nparticles/(8*cfg.dataset.rho))**(1/3)
    else:
        B=cfg.dataset.ncellx*cfg.dataset.cell_len/2
    boxlength=2*B
    N=cfg.dataset.nparticles* cfg.dataset.dim
    logging.basicConfig(level=logging.DEBUG)
    logger = logging.getLogger(__name__)  
    if cfg.prior.type=="lattice":
        prior = EinsteinCrystal(cfg.prior.lattice_dir, alpha=cfg.prior.alpha,device=cfg.device)
    elif cfg.prior.type=="normal":
        prior = MultivariateNormal(torch.zeros(N).to(cfg.device), 0.1*B*torch.eye(N).to(cfg.device))
    elif cfg.prior.type=="gaussian_mix":
        prior = systems.GaussianMixture(cfg.prior.centers,cfg.prior.vars,cfg.dataset.nparticles,cfg.dataset.dim)
    if cfg.flow.type=="RealNVP":
        flows = [eval(cfg.flow.type)(dim=N,hidden_dim=cfg.flow.hidden_dim) for _ in range(cfg.flow.nlayers)]
    elif cfg.flow.type=="NSF_AR":
        flows = [eval(cfg.flow.type)(dim=N, K=cfg.flow.nsplines, B=B,hidden_dim=cfg.flow.hidden_dim,device=cfg.device) for _ in range(cfg.flow.nlayers)]
    elif cfg.flow.type=="NSF_CL":
        x = [[0],[1],[2],[0,1],[1,2],[0,2]]
        mask= sum([x for _ in range(cfg.flow.nlayers//6+1)], [])[:cfg.flow.nlayers]
        flows = [eval(cfg.flow.type)(size=cfg.dataset.nparticles,dim=3, K=cfg.flow.nsplines, B=B,hidden_dim=cfg.flow.hidden_dim, mask=mask[i],device=cfg.device) for i in range(cfg.flow.nlayers)]
    model = NormalizingFlowModel(prior, flows,cfg.device).to(cfg.device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.train_parameters.learning_rate)
    scheduler = torch. optim.lr_scheduler.ExponentialLR(optimizer, cfg.train_parameters.lr_scheduler_gamma)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.train_parameters.max_epochs)
    if cfg.dataset.training_dir is not None:
        training_data = util.load_position(cfg.dataset.training_dir).to(cfg.device)
    else:
        training_data=None
    if not(os.path.exists(cfg.output.model_dir)):
        os.mkdir(cfg.output.model_dir)
    if not(os.path.exists(cfg.output.training_dir)):
        os.mkdir(cfg.output.training_dir)
    if not(os.path.exists(cfg.output.testing_dir)):
        os.mkdir(cfg.output.testing_dir)
    if cfg.dataset.potential=="GaussianMixture":
        potential=systems.GaussianMixture(cfg.dataset.centers,cfg.dataset.vars,cfg.dataset.nparticles,torch.tensor(cfg.dataset.centers).shape[1])
        
    return model,optimizer,scheduler,training_data,logger,boxlength,potential

In [3]:
def train(cfg,model,optimizer,scheduler,training_data,logger,boxlength,potential):
    losses=[]
    max_logprob=-1
    lamb=0.3
    for i in range(cfg.train_parameters.max_epochs):
        optimizer.zero_grad()
        x = potential.sample(cfg.train_parameters.batch_size,flatten=True).to(cfg.device)
        z, prior_logprob, log_det = model(x)
        #x1, log_det1 =model.inverse(z)
        #print("check the map is invertible", torch.linalg.norm(x1-x), torch.linalg.norm(log_det1+log_det))
        logprob = prior_logprob + log_det
        forward_loss=-torch.mean(logprob)+torch.mean(potential.log_prob(x))
        if i>10000:
            loss = forward_loss*lamb + (1-lamb)*reverseKL(cfg,model, cfg.train_parameters.batch_size)
        else:
            loss = forward_loss
        losses.append(loss.mean().data)
        loss.backward()
        optimizer.step()
        scheduler.step()
        if i % cfg.train_parameters.output_freq == 0:
            logger.info(f"Iter: {i}\t" +
                        f"Loss: {loss.mean().data:.2f}\t" +
                        f"Logprob: {logprob.mean().data:.2f}\t" +
                        f"Prior: {prior_logprob.mean().data:.2f}\t" +
                        f"LogDet: {log_det.mean().data:.2f}")
            '''
            samples,_,z = model.sample(1)
            util.write_coord(cfg.output.training_dir+"generated_configs.xyz",samples,cfg.dataset.nparticles)
            '''
            if (i>200) and (-forward_loss>max_logprob):
                max_logprob=-forward_loss
                torch.save({"model":model.state_dict(),"optim": optimizer.state_dict(),
                            "loss":losses},cfg.output.model_dir+cfg.dataset.name+'%d.pth'% (i//cfg.train_parameters.output_freq))

In [4]:
def read_input(dir):
    cfg = get_cfg_defaults()
    cfg.merge_from_file(dir)
    cfg.freeze()
    print(cfg)
    return cfg

In [5]:
def reverseKL(cfg,model,nsamples):
    z = model.prior.sample((nsamples,))
    x, log_det = model.inverse(z)
    log_prob = model.prior.log_prob(z)-log_det
    return -torch.mean(potential.logprob(x))+torch.mean(log_prob)

In [5]:
name="Gaussian_rnvp"
cfg=read_input("input/%s.yaml"%name)
model,optimizer,scheduler,training_data,logger,boxlength,potential = setup_model(cfg)

dataset:
  cell_len: 1
  centers: [[0.5, 0.5]]
  dim: 2
  epsilon: 1.0
  kT: 1.0
  name: Gaussian_rnvp
  ncellx: 8
  ncelly: 8
  ncellz: 8
  nparticles: 20
  periodic: True
  potential: GaussianMixture
  rho: None
  sigma: 1.0
  testing_dir: None
  training_dir: None
  vars: [[0.36]]
device: cpu
flow:
  hidden_dim: 80
  nlayers: 4
  nsplines: 32
  type: RealNVP
output:
  model_dir: saved_models/
  testing_dir: testing/gaussian/
  training_dir: training/gaussian/
prior:
  alpha: 100
  centers: [[-0.5, -0.5]]
  lattice_dir: structures/ref.xyz
  type: gaussian_mix
  vars: [[0.25]]
train_parameters:
  batch_size: 60
  learning_rate: 0.005
  lr_scheduler_gamma: 0.999
  max_epochs: 3000
  output_freq: 100
  scheduler: exponential


In [6]:
train(cfg,model,optimizer,scheduler,training_data,logger,boxlength,potential)

INFO:__main__:Iter: 0	Loss: 122.22	Logprob: -158.87	Prior: -161.38	LogDet: 2.51
INFO:__main__:Iter: 100	Loss: 1.56	Logprob: -38.35	Prior: -28.57	LogDet: -9.78
INFO:__main__:Iter: 200	Loss: 1.08	Logprob: -38.11	Prior: -29.93	LogDet: -8.18
INFO:__main__:Iter: 300	Loss: 1.05	Logprob: -38.33	Prior: -30.10	LogDet: -8.23
INFO:__main__:Iter: 400	Loss: 0.43	Logprob: -37.34	Prior: -29.86	LogDet: -7.48
INFO:__main__:Iter: 500	Loss: 0.77	Logprob: -37.62	Prior: -29.81	LogDet: -7.81
INFO:__main__:Iter: 600	Loss: 0.59	Logprob: -37.06	Prior: -29.40	LogDet: -7.66
INFO:__main__:Iter: 700	Loss: 0.18	Logprob: -37.30	Prior: -29.78	LogDet: -7.52
INFO:__main__:Iter: 800	Loss: 0.23	Logprob: -36.44	Prior: -28.63	LogDet: -7.81
INFO:__main__:Iter: 900	Loss: 0.29	Logprob: -36.41	Prior: -28.75	LogDet: -7.66
INFO:__main__:Iter: 1000	Loss: 0.42	Logprob: -36.71	Prior: -29.15	LogDet: -7.56
INFO:__main__:Iter: 1100	Loss: 0.31	Logprob: -36.03	Prior: -28.75	LogDet: -7.28
INFO:__main__:Iter: 1200	Loss: 0.18	Logprob: -37.

In [1]:

import matplotlib.pyplot as plt
def plot(x):
    x=x.numpy().reshape((-1,2))
    plt.scatter(x[:,0],x[:,1])
    plt.show()
    plt.hist(x[:,0])
    plt.show()
    plt.hist(x[:,1])
    plt.show()
    plt.close()

In [2]:
def generate_from_nf(model, prior, nsamples=50):
    x, log_det ,z = model.sample(nsamples)
    z=prior.sample((nsamples,))
    x, log_det = model.inverse(z)
    log_px=prior.log_prob(z)-log_det
    return x.data, log_px.data

In [17]:
def compute_fe_diff(cfg,model,nsamples):
        nf = torch.load("trained_models_new/%s.pth"%name,map_location='cpu')
        #nf = torch.load("saved_models/%s14.pth"%name,map_location='cpu')
        np.savetxt(cfg.output.testing_dir+"loss_%s.dat"%cfg.dataset.name,torch.Tensor(nf["loss"]).cpu().numpy())
        model.load_state_dict(nf["model"],strict=False)
        model=model.to(cfg.device)
        traj0,q00=generate_from_nf(model,model.prior, nsamples)
        #plot(traj0)
        q00=q00.cpu().numpy()
        #traj0=traj0.cpu().reshape(-1,cfg.dataset.nparticles,cfg.dataset.dim)
        q01=potential.log_prob(traj0).detach().cpu().numpy()
        Q=[]
        Q.append(np.transpose(np.vstack((q00,q01))))
        traj1=potential.sample(nsamples)
        #plot(traj1)
        z,_,log_det=model.forward(traj1.reshape(len(traj1),-1))
        log_det=0
        q10=model.prior.log_prob(z)-log_det
        q10=q10.detach().cpu().numpy()
        q11=potential.log_prob(traj1).detach().cpu().numpy()
        q10=q11
        Q.append(np.transpose(np.vstack((q10,q11))))
        with open(cfg.output.testing_dir+"Q0_%s.dat"%cfg.dataset.name, "w") as f:
                np.savetxt(f, Q[0])
        with open(cfg.output.testing_dir+"Q1_%s.dat"%cfg.dataset.name,"w"):
                np.savetxt(cfg.output.testing_dir+"Q1_%s.dat"%cfg.dataset.name,Q[1])
        to_subtract_0=np.min(Q[0][:,0])
        to_subtract_1=np.min(Q[0][:,1])
        Q[0][:,0]-=to_subtract_0
        Q[0][:,1]-=to_subtract_1
        Q[1][:,1]-=to_subtract_1
        Q[1][:,0]-=to_subtract_0
        bar=(to_subtract_0-to_subtract_1+BAR(Q[0][:,0]-Q[0][:,1],-Q[1][:,0]+Q[1][:,1]))/cfg.dataset.nparticles*cfg.dataset.kT
        md = (to_subtract_0-to_subtract_1)/cfg.dataset.nparticles*cfg.dataset.kT-np.log(np.mean(np.exp(Q[1][:,1]-Q[1][:,0])))/cfg.dataset.nparticles*cfg.dataset.kT
        nf = -((to_subtract_1-to_subtract_0)/cfg.dataset.nparticles*cfg.dataset.kT-np.log(np.mean(np.exp(Q[0][:,0]-Q[0][:,1])))/cfg.dataset.nparticles*cfg.dataset.kT)
        c = solver([np.exp(Q[0]),np.exp(Q[1])],niter=40).norm_const()
        emus=(to_subtract_0-to_subtract_1+np.log(c[0]))/cfg.dataset.nparticles*cfg.dataset.kT
        return bar, md, nf, emus

In [4]:
def plot_Q(cfg,Q):
    fig,(ax1,ax2)=plt.subplots(1,2,sharey=True,figsize=(12,6),tight_layout=True)
    ax1.plot(Q[0][:,0], Q[0][:,1],'.',color="darkgray")
    ax1.set_title("trajectory generated by NF")
    ax2.plot(Q[1][:,0], Q[1][:,1],'.',color="darkgray")
    ax2.set_title("trajectory from MD simulation")
    fig.supxlabel("logpx from NF")
    fig.supylabel("-potential (kT)")
    fig.savefig(cfg.output.testing_dir+"Q_%s.png"%cfg.dataset.name)
    plt.show()
    plt.close()

In [5]:
def plot_trend(cfg,x,y,err,xlabel=None,title=None,log_base=None):
    plt.scatter(x,y,marker='.')
    print(x,y,err)
    plt.errorbar(x, y,yerr=err,fmt=".")
    if log_base is not None:
        plt.xscale('log',base=log_base)
    if xlabel is not None:
        plt.xlabel(xlabel)
    plt.ylabel("FE difference")
    if title is not None:
        plt.title(title)
    plt.plot(x,np.zeros(len(x)),label="y=0")
    plt.legend()
    plt.savefig("%s.png"%cfg.output.testing_dir+title)
    plt.show()
    plt.close()

In [24]:
def size_dependence(cfg,model,nsamples,ntrials):
    fe_bar_list=[]
    fe_bar_err_list=[]
    fe_md_list=[]
    fe_md_err_list=[]
    fe_nf_list=[]
    fe_nf_err_list=[]
    fe_emus_list=[]
    fe_emus_err_list=[]
    for i in nsamples:
        fe_bar=[]
        fe_md=[]
        fe_nf=[]
        fe_emus=[]
        for _ in range(ntrials):
            bar,md,nf,emus = compute_fe_diff(cfg,model,i)
            fe_bar.append(bar)
            fe_md.append(md)
            fe_nf.append(nf)
            fe_emus.append(emus)
        fe_bar_list.append(np.mean(np.array(fe_bar)))
        fe_bar_err_list.append(np.std(np.array(fe_bar)))
        fe_md_list.append(np.mean(np.array(fe_md)))
        fe_md_err_list.append(np.std(np.array(fe_md)))
        fe_nf_list.append(np.mean(np.array(fe_nf)))
        fe_nf_err_list.append(np.std(np.array(fe_nf)))
        fe_emus_list.append(np.mean(np.array(fe_emus)))
        fe_emus_err_list.append(np.std(np.array(fe_emus)))
    #plot_trend(cfg, nsamples,np.array(fe_bar_list),np.array(fe_bar_err_list),"number of samplses","bar_estimate",log_base=2)
    #plot_trend(cfg, nsamples,np.array(fe_md_list),np.array(fe_md_err_list),"number of samplses","simple_estimate_from_md_data",log_base=2)
    #plot_trend(cfg, nsamples,np.array(fe_nf_list),np.array(fe_nf_err_list),"number of samplses","simple_estimate_from_generated_data",log_base=2)
    return np.array(fe_bar_list),np.array(fe_bar_err_list), np.array(fe_md_list),np.array(fe_md_err_list), np.array(fe_nf_list),np.array(fe_nf_err_list),np.array(fe_emus_list),np.array(fe_emus_err_list)

In [26]:
nsamples=2**(np.arange(0,8,2)+1)
ntrials=20
bar, bar_err, md, md_err, nf, nf_err, emus, emus_err=size_dependence(cfg,model,nsamples,ntrials)

In [21]:
plot_trend(cfg, nsamples,np.array(bar),np.array(bar_err),"number of samplses","bar_estimate",log_base=2)
plot_trend(cfg, nsamples,np.array(md),np.array(md_err),"number of samplses","simple_estimate_from_md_data",log_base=2)
plot_trend(cfg, nsamples,np.array(nf),np.array(nf_err),"number of samplses","simple_estimate_from_generated_data",log_base=2)
plot_trend(cfg, nsamples,np.array(emus),np.array(emus_err),"number of samplses","emus_estimate",log_base=2)

[]
